In [0]:
storage_account_name = "aparnastshellenergyuks"
storage_account_key = dbutils.secrets.get(
    scope="aparna-kv-shellenergy-scope",
    key="aparna-adls-storage-account-key"
)
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"

# Event Hub connection string - same one used by 05_stream_generator.py
eventhub_connection_string = dbutils.secrets.get(
    scope="aparna-kv-shellenergy-scope",
    key="aparna-eventhub-connection-string"
)

print("Auth configured.")

In [0]:
eventhub_namespace = "aparna-evhns-shellenergy-uk"
bootstrap_servers = f"{eventhub_namespace}.servicebus.windows.net:9093"

# SASL config using the namespace connection string - this is the standard pattern
# for connecting to Event Hub's Kafka-compatible endpoint
eh_sasl = (
    f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" password="{eventhub_connection_string}";'
)

kafka_options_meter = {
    "kafka.bootstrap.servers": bootstrap_servers,
    "subscribe": "aparna-meter-readings-stream",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": eh_sasl,
    "kafka.request.timeout.ms": "60000",
    "kafka.session.timeout.ms": "60000",
    "startingOffsets": "earliest",  # read from the beginning, including events sent during earlier testing
    "failOnDataLoss": "false",
}

print("Kafka options configured for meter readings stream.")
print(f"Bootstrap servers: {bootstrap_servers}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from pyspark.sql.functions import col, from_json

meter_reading_schema = StructType([
    StructField("event_type", StringType()),
    StructField("site_id", StringType()),
    StructField("mpan", StringType()),
    StructField("interval_start", StringType()),  # parsed as string first, cast to timestamp after
    StructField("kwh_interval", DoubleType()),
    StructField("meter_status", StringType()),
    StructField("ingested_at", StringType()),
])

df_meter_stream_raw = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options_meter)
    .load()
)

df_meter_parsed = (
    df_meter_stream_raw
    .selectExpr("CAST(value AS STRING) as json_str", "timestamp as kafka_timestamp", "offset", "partition")
    .withColumn("parsed", from_json(col("json_str"), meter_reading_schema))
    .select(
        col("parsed.event_type"),
        col("parsed.site_id"),
        col("parsed.mpan"),
        col("parsed.interval_start").cast(TimestampType()).alias("interval_start"),
        col("parsed.kwh_interval"),
        col("parsed.meter_status"),
        col("parsed.ingested_at").cast(TimestampType()).alias("ingested_at"),
        col("kafka_timestamp"),
        col("offset"),
        col("partition"),
    )
)

print("Meter reading stream schema parsed.")
df_meter_parsed.printSchema()

In [0]:
wholesale_price_schema = StructType([
    StructField("event_type", StringType()),
    StructField("interval_start", StringType()),
    StructField("price_gbp_per_mwh", DoubleType()),
    StructField("market_period", StringType()),
    StructField("ingested_at", StringType()),
])

df_price_stream_raw = (
    spark.readStream
    .format("kafka")
    .options(**{**kafka_options_meter, "subscribe": "aparna-wholesale-price-stream"})
    .load()
)

df_price_parsed = (
    df_price_stream_raw
    .selectExpr("CAST(value AS STRING) as json_str", "timestamp as kafka_timestamp", "offset", "partition")
    .withColumn("parsed", from_json(col("json_str"), wholesale_price_schema))
    .select(
        col("parsed.event_type"),
        col("parsed.interval_start").cast(TimestampType()).alias("interval_start"),
        col("parsed.price_gbp_per_mwh"),
        col("parsed.market_period"),
        col("parsed.ingested_at").cast(TimestampType()).alias("ingested_at"),
        col("kafka_timestamp"),
        col("offset"),
        col("partition"),
    )
)

print("Wholesale price stream schema parsed.")
df_price_parsed.printSchema()

In [0]:
streaming_bronze_path = f"{bronze_path}/streaming"
checkpoint_base = f"{streaming_bronze_path}/_checkpoints"

meter_write_query = (
    df_meter_parsed.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_base}/meter_readings_stream/")
    .outputMode("append")
    .trigger(processingTime="10 seconds")
    .start(f"{streaming_bronze_path}/meter_readings_stream/")
)

price_write_query = (
    df_price_parsed.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_base}/wholesale_price_stream/")
    .outputMode("append")
    .trigger(processingTime="10 seconds")
    .start(f"{streaming_bronze_path}/wholesale_price_stream/")
)

print("Both streaming writes started.")
print(f"Meter readings -> {streaming_bronze_path}/meter_readings_stream/")
print(f"Wholesale prices -> {streaming_bronze_path}/wholesale_price_stream/")

In [0]:
df_check = spark.read.format("delta").load(f"{streaming_bronze_path}/meter_readings_stream/")
print(f"Meter readings stream Bronze row count: {df_check.count()}")
display(df_check.orderBy("kafka_timestamp", ascending=False).limit(5))

In [0]:
df_check_price = spark.read.format("delta").load(f"{streaming_bronze_path}/wholesale_price_stream/")
print(f"Wholesale price stream Bronze row count: {df_check_price.count()}")
display(df_check_price.orderBy("kafka_timestamp", ascending=False).limit(5))

In [0]:
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
streaming_silver_path = f"{silver_path}/streaming"

# Read the Bronze streaming Delta table AS A STREAM (not a static read) - 
# this lets Silver continuously pick up new micro-batches as Bronze writes them
df_meter_bronze_stream = spark.readStream.format("delta").load(f"{streaming_bronze_path}/meter_readings_stream/")

from pyspark.sql.functions import col

df_meter_silver_stream = (
    df_meter_bronze_stream
    .filter(col("site_id").isNotNull())  # basic validation - drop anything that failed to parse
    .filter(col("kwh_interval") >= 0)     # sanity check - no negative consumption
    .dropDuplicates(["site_id", "interval_start", "offset", "partition"])  # dedup safeguard
)

meter_silver_query = (
    df_meter_silver_stream.writeStream
    .format("delta")
    .option("checkpointLocation", f"{streaming_silver_path}/_checkpoints/meter_readings_stream/")
    .outputMode("append")
    .trigger(processingTime="15 seconds")
    .start(f"{streaming_silver_path}/meter_readings_stream/")
)

print("Streaming Silver write started for meter readings.")

In [0]:
df_price_bronze_stream = spark.readStream.format("delta").load(f"{streaming_bronze_path}/wholesale_price_stream/")

df_price_silver_stream = (
    df_price_bronze_stream
    .filter(col("price_gbp_per_mwh").isNotNull())
    .filter(col("price_gbp_per_mwh") > 0)  # sanity check - no zero/negative prices
    .dropDuplicates(["interval_start", "offset", "partition"])
)

price_silver_query = (
    df_price_silver_stream.writeStream
    .format("delta")
    .option("checkpointLocation", f"{streaming_silver_path}/_checkpoints/wholesale_price_stream/")
    .outputMode("append")
    .trigger(processingTime="15 seconds")
    .start(f"{streaming_silver_path}/wholesale_price_stream/")
)

print("Streaming Silver write started for wholesale prices.")

In [0]:
import time
time.sleep(20)

df_meter_silver_check = spark.read.format("delta").load(f"{streaming_silver_path}/meter_readings_stream/")
print(f"Silver meter readings stream row count: {df_meter_silver_check.count()}")

df_price_silver_check = spark.read.format("delta").load(f"{streaming_silver_path}/wholesale_price_stream/")
print(f"Silver wholesale price stream row count: {df_price_silver_check.count()}")

display(df_meter_silver_check.orderBy("kafka_timestamp", ascending=False).limit(5))

In [0]:
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"
streaming_gold_path = f"{gold_path}/streaming"

# Static reference( stream-static join): contracted daily run-rate per site (reuse the same calc as batch)
df_contracts_static = spark.read.format("delta").load(f"{silver_path}/batch/contracts/")
df_contracts_daily_rate_static = (
    df_contracts_static
    .withColumn("contracted_daily_kwh", col("contracted_annual_kwh") / 365.25)
    .select("site_id", "contract_id", "start_date", "end_date", "contracted_daily_kwh")
)

# Read Silver streaming meter readings AS A STREAM
df_meter_silver_stream_read = spark.readStream.format("delta").load(f"{streaming_silver_path}/meter_readings_stream/")

from datetime import date

today = date.today().isoformat()  # plain Python value, computed once, not a Spark expression

# Filter to currently-active contracts BEFORE this becomes part of any streaming join
df_contracts_daily_rate_static = (
    df_contracts_static
    .withColumn("contracted_daily_kwh", col("contracted_annual_kwh") / 365.25)
    .filter((col("start_date") <= today) & (col("end_date") >= today))
    .select("site_id", "contract_id", "start_date", "end_date", "contracted_daily_kwh")
)

print(f"Active contracts as of {today}: {df_contracts_daily_rate_static.count()}")

# Stream-static join: match each live reading to its active contract
from pyspark.sql.functions import current_date

df_meter_with_contract = (
    df_meter_silver_stream_read.alias("m")
    .join(
        df_contracts_daily_rate_static.alias("c"),
        (col("m.site_id") == col("c.site_id")),
        how="left"
    )
    .select(
        col("m.site_id"), col("m.interval_start"), col("m.kwh_interval"),
        col("c.contract_id"), col("c.contracted_daily_kwh")
    )
)

print("Stream-static join defined (meter readings <-> contracts).")

In [0]:
from pyspark.sql.functions import expr, round as spark_round

# Read Silver streaming wholesale prices AS A STREAM
df_price_silver_stream_read = spark.readStream.format("delta").load(f"{streaming_silver_path}/wholesale_price_stream/")

# Stream-stream joins require watermarks on both sides, so Spark knows how long
# to hold state waiting for a match before giving up - 10 minutes is generous
# given our events arrive every few seconds in simulated time
df_meter_watermarked = df_meter_with_contract.withWatermark("interval_start", "10 minutes")
df_price_watermarked = (
    df_price_silver_stream_read
    .withWatermark("interval_start", "10 minutes")
    .select(col("interval_start").alias("price_interval_start"), "price_gbp_per_mwh")
)

df_live_exposure = (
    df_meter_watermarked.alias("m")
    .join(
        df_price_watermarked.alias("p"),
        expr("m.interval_start = p.price_interval_start"),
        how="inner"
    )
    .withColumn(
        "exposure_kwh",
        spark_round(col("kwh_interval") - (col("contracted_daily_kwh") / 48), 4)  # /48 = HH share of daily rate
    )
    .withColumn(
        "live_exposure_gbp",
        spark_round((col("exposure_kwh") / 1000) * col("price_gbp_per_mwh"), 4)
    )
    .select(
        "site_id", "interval_start", "kwh_interval", "contracted_daily_kwh",
        "exposure_kwh", "price_gbp_per_mwh", "live_exposure_gbp"
    )
)

print("Live exposure calculation defined (meter <-> contract <-> live price).")

In [0]:
live_exposure_query = (
    df_live_exposure.writeStream
    .format("delta")
    .option("checkpointLocation", f"{streaming_gold_path}/_checkpoints/live_exposure/")
    .outputMode("append")
    .trigger(processingTime="15 seconds")
    .start(f"{streaming_gold_path}/live_exposure/")
)

print("Streaming Gold write started for live exposure.")

In [0]:
import time
time.sleep(20)

df_exposure_check = spark.read.format("delta").load(f"{streaming_gold_path}/live_exposure/")
print(f"Live exposure row count: {df_exposure_check.count()}")
display(df_exposure_check.orderBy("interval_start", ascending=False).limit(10))

In [0]:
%skip
## Utility: stop all active streams
# Run this before re-starting the pipeline fresh, or before closing the notebook.
for s in spark.streams.active:
    s.stop()
print("All test streams stopped.")

In [0]:
%skip
df_meter_stream_raw = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options_meter)
    .load()
)

# Write to an in-memory table for quick inspection (NOT a production pattern - 
# just for testing the connection works before building the real Bronze write)
query = (
    df_meter_stream_raw
    .writeStream
    .format("memory")
    .queryName("meter_stream_test")
    .outputMode("append")
    .start()
)

import time
time.sleep(15)  # give it a few seconds to pull whatever's available
display(spark.sql("SELECT * FROM meter_stream_test LIMIT 10"))

In [0]:
%skip
# Stop any existing streaming query first
for s in spark.streams.active:
    s.stop()
print("All active streams stopped.")

In [0]:
%skip
df_meter_stream_raw = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options_meter)
    .load()
)

query2 = (
    df_meter_stream_raw
    .writeStream
    .format("memory")
    .queryName("meter_stream_test_v2")
    .outputMode("append")
    .start()
)

In [0]:
%skip
import time
time.sleep(10)
display(spark.sql("SELECT COUNT(*) as row_count FROM meter_stream_test_v2"))
display(spark.sql("SELECT * FROM meter_stream_test_v2 LIMIT 5"))


query2.stop()
print("Test query stopped.")

In [0]:
print("BEFORE counts:")
print("Bronze meter:", spark.read.format("delta").load(f"{streaming_bronze_path}/meter_readings_stream/").count())
print("Bronze price:", spark.read.format("delta").load(f"{streaming_bronze_path}/wholesale_price_stream/").count())
print("Silver meter:", spark.read.format("delta").load(f"{streaming_silver_path}/meter_readings_stream/").count())
print("Silver price:", spark.read.format("delta").load(f"{streaming_silver_path}/wholesale_price_stream/").count())
print("Gold exposure:", spark.read.format("delta").load(f"{streaming_gold_path}/live_exposure/").count())

In [0]:
import time
time.sleep(30)  # give all 3 layers time to process their micro-batches

print("AFTER counts:")
print("Bronze meter:", spark.read.format("delta").load(f"{streaming_bronze_path}/meter_readings_stream/").count())
print("Bronze price:", spark.read.format("delta").load(f"{streaming_bronze_path}/wholesale_price_stream/").count())
print("Silver meter:", spark.read.format("delta").load(f"{streaming_silver_path}/meter_readings_stream/").count())
print("Silver price:", spark.read.format("delta").load(f"{streaming_silver_path}/wholesale_price_stream/").count())
print("Gold exposure:", spark.read.format("delta").load(f"{streaming_gold_path}/live_exposure/").count())

In [0]:
%skip
print(live_exposure_query.lastProgress)

'''for s in spark.streams.active:
    print(s.id, "->", s.lastProgress.get("sink", {}).get("description", "no description"))
    print("   numInputRows last batch:", s.lastProgress.get("sources", [{}])[0].get("numInputRows"))
    print("   watermark:", s.lastProgress.get("eventTime", {}).get("watermark"))'''

In [0]:
%skip
#clears its internal watermark state
live_exposure_query.stop()

In [0]:
%skip
# Stop the Gold query first if it's running
for s in spark.streams.active:
    if "live_exposure" in s.lastProgress.get("sink", {}).get("description", ""):
        s.stop()

# Delete the checkpoint folder entirely - this is what's preserving the stuck watermark
dbutils.fs.rm(f"{streaming_gold_path}/_checkpoints/live_exposure/", recurse=True)
print("Gold checkpoint deleted - next start will be genuinely fresh.")